# 3. Nutrition Estimation Module

This module estimates nutrition information based on:
- Ingredient type
- Bounding box size (estimated weight)
- USDA FoodData Central database

**Usage:**
```python
%run 3_nutrition_estimation.ipynb
nutrition = estimate_nutrition(ingredient, bbox_width, bbox_height)
```

In [1]:
import json
from pathlib import Path
from typing import Dict

print("✓ Imports loaded")

✓ Imports loaded


In [2]:
# Paths
PROJECT_ROOT = Path.cwd().parent.parent.parent
DATA_DIR = PROJECT_ROOT / "data"
NUTRITION_JSON = DATA_DIR / "nutrition_lookup_full.json"

print(f"✓ Data directory: {DATA_DIR}")

✓ Data directory: c:\Users\Champion\Documents\GitHub\cAIuldron\data


In [3]:
# Load nutrition database
if NUTRITION_JSON.exists():
    with open(NUTRITION_JSON, 'r', encoding='utf-8') as f:
        NUTRITION_DB = json.load(f)
    print(f"✓ Nutrition database loaded: {len(NUTRITION_DB)} ingredients")
else:
    print(f"⚠ Nutrition database not found, using fallback...")
    NUTRITION_DB = {
        'chicken breast': {'calories': 165, 'protein_g': 31, 'fat_g': 3.6, 'carbs_g': 0},
        'chicken': {'calories': 239, 'protein_g': 27, 'fat_g': 14, 'carbs_g': 0},
        'beef': {'calories': 250, 'protein_g': 26, 'fat_g': 15, 'carbs_g': 0},
        'salmon': {'calories': 208, 'protein_g': 20, 'fat_g': 13, 'carbs_g': 0},
        'tomato': {'calories': 18, 'protein_g': 0.9, 'fat_g': 0.2, 'carbs_g': 3.9},
        'potato': {'calories': 77, 'protein_g': 2, 'fat_g': 0.1, 'carbs_g': 17},
        'egg': {'calories': 155, 'protein_g': 13, 'fat_g': 11, 'carbs_g': 1.1},
        'rice': {'calories': 130, 'protein_g': 2.7, 'fat_g': 0.3, 'carbs_g': 28},
    }
    print(f"✓ Fallback database loaded: {len(NUTRITION_DB)} ingredients")

✓ Nutrition database loaded: 525 ingredients


In [4]:
# Typical weights for common ingredients (in grams)
TYPICAL_WEIGHTS = {
    'chicken breast': 200, 'chicken': 150, 'beef': 200,
    'salmon': 150, 'tomato': 120, 'potato': 180,
    'egg': 50, 'rice': 150
}

print("✓ Typical weights loaded")

✓ Typical weights loaded


In [5]:
def estimate_nutrition(ingredient: str, bbox_width: int, bbox_height: int,
                      image_width: int = 640, image_height: int = 640) -> Dict:
    """Estimate nutrition from bounding box"""
    ingredient_lower = ingredient.lower()
    
    # Estimate weight from bounding box area
    typical_weight = TYPICAL_WEIGHTS.get(ingredient_lower, 150)
    bbox_area = bbox_width * bbox_height
    image_area = image_width * image_height
    area_ratio = bbox_area / image_area
    size_multiplier = (area_ratio / 0.25) ** 0.7
    estimated_weight = typical_weight * size_multiplier
    
    # Determine serving size based on ingredient type
    if any(m in ingredient_lower for m in ['chicken', 'beef', 'pork', 'salmon', 'fish']):
        serving_size = 120
    elif any(v in ingredient_lower for v in ['potato', 'tomato', 'vegetable']):
        serving_size = 100
    else:
        serving_size = 100
    
    servings = max(1, round(estimated_weight / serving_size * 2) / 2)
    g_per_serving = estimated_weight / servings
    
    # Find nutrition data
    nutrition_base = None
    if ingredient in NUTRITION_DB:
        nutrition_base = NUTRITION_DB[ingredient]
    else:
        for key in NUTRITION_DB.keys():
            if key.lower() in ingredient_lower or ingredient_lower in key.lower():
                nutrition_base = NUTRITION_DB[key]
                break
    
    if not nutrition_base:
        return {'success': False, 'error': f'No nutrition data for {ingredient}'}
    
    # Calculate nutrition per serving
    multiplier = g_per_serving / 100
    calories = nutrition_base['calories'] * multiplier
    
    return {
        'success': True,
        'weight_g': round(estimated_weight, 1),
        'servings': int(servings) if servings.is_integer() else servings,
        'per_serving': {
            'weight_g': round(g_per_serving, 1),
            'calories': round(calories, 0),
            'calories_range': f"{round(calories*0.8, 0):.0f}-{round(calories*1.2, 0):.0f} kcal",
            'protein_g': round(nutrition_base['protein_g'] * multiplier, 1),
            'fat_g': round(nutrition_base['fat_g'] * multiplier, 1),
            'carbs_g': round(nutrition_base['carbs_g'] * multiplier, 1)
        }
    }

print("✓ Nutrition estimation function defined")

✓ Nutrition estimation function defined


## Test Nutrition Estimation

Uncomment to test:

In [8]:
nutrition = estimate_nutrition('chicken breast', 200, 200)
if nutrition['success']:
   print(f"Weight: {nutrition['weight_g']}g")
   print(f"Servings: {nutrition['servings']}")
   print(f"Calories per serving: {nutrition['per_serving']['calories']} kcal")

Weight: 103.6g
Servings: 1
Calories per serving: 272.0 kcal


---

**Module exports:**
- `NUTRITION_DB` (dict)
- `TYPICAL_WEIGHTS` (dict)
- `estimate_nutrition()` (function)